# Deep Q-Networks (DQN): Q-learning with a neural net

_Reinforcement Learning_

**Swap Q-learning's giant table for a neural network, then add experience replay and a target network so training does not blow up.**

Tabular Q-learning (see ai-q-learning) stores one number $Q(s,a)$ for every
       state-action pair. Real problems &mdash; pixels, sensors &mdash; have astronomically many states,
       so the table cannot exist.

       A Deep Q-Network replaces the table with a neural network
       $\hat Q(s,a;\theta)$: it reads the state $s$ and outputs an estimated value for each action $a$,
       using weights $\theta$ (Greek "theta"). The hat on $\hat Q$ marks it as an APPROXIMATION, not the
       exact value. The network generalizes: states that look similar get similar values, for free.

---

This notebook is a practice scaffold for the **Deep Q-Networks (DQN): Q-learning with a neural net** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — gymnasium + PyTorch (runs in Colab)

In [ ]:
# DQN on CartPole-v1 — Colab:  !pip install gymnasium torch
import random, collections
import numpy as np, torch, torch.nn as nn, gymnasium as gym

env = gym.make("CartPole-v1")
n_obs = env.observation_space.shape[0]   # 4: position, velocity, angle, angular velocity
n_act = env.action_space.n               # 2: push left / push right
device = "cuda" if torch.cuda.is_available() else "cpu"

def make_q():                            # small MLP Q-network: state -> one Q value per action
    return nn.Sequential(
        nn.Linear(n_obs, 128), nn.ReLU(),
        nn.Linear(128, 128),   nn.ReLU(),
        nn.Linear(128, n_act)).to(device)

q      = make_q()                        # online network, weights theta
q_targ = make_q(); q_targ.load_state_dict(q.state_dict())   # target net, weights theta-minus (frozen copy)
opt    = torch.optim.Adam(q.parameters(), lr=1e-3)
buffer = collections.deque(maxlen=50_000)    # experience replay buffer of (s,a,r,s',done)
gamma, batch_size, target_sync = 0.99, 64, 500

def act(state, eps):                     # epsilon-greedy: explore with prob eps, else greedy
    if random.random() < eps:
        return env.action_space.sample()
    with torch.no_grad():
        s = torch.as_tensor(state, dtype=torch.float32, device=device)
        return int(q(s).argmax().item())

def learn():                             # one gradient step on a random minibatch
    if len(buffer) < batch_size:
        return
    batch = random.sample(buffer, batch_size)           # random sample breaks correlation
    s, a, r, s2, d = map(np.array, zip(*batch))
    s   = torch.as_tensor(s,  dtype=torch.float32, device=device)
    s2  = torch.as_tensor(s2, dtype=torch.float32, device=device)
    a   = torch.as_tensor(a,  dtype=torch.int64,   device=device)
    r   = torch.as_tensor(r,  dtype=torch.float32, device=device)
    d   = torch.as_tensor(d,  dtype=torch.float32, device=device)
    q_sa = q(s).gather(1, a.unsqueeze(1)).squeeze(1)     # Q(s,a;theta)
    with torch.no_grad():                               # target uses FROZEN theta-minus
        max_next = q_targ(s2).max(dim=1).values          # max_a' Q(s',a'; theta-minus)
        y = r + gamma * max_next * (1.0 - d)             # Bellman TD target; 0 bootstrap if done
    loss = nn.functional.mse_loss(q_sa, y)               # mean squared TD error
    opt.zero_grad(); loss.backward(); opt.step()

steps = 0
for ep in range(400):
    state, _ = env.reset()
    eps = max(0.02, 1.0 - ep / 200)                      # anneal exploration high -> low
    ep_return, done = 0.0, False
    while not done:
        action = act(state, eps)
        nxt, reward, term, trunc, _ = env.step(action)
        done = term or trunc
        buffer.append((state, action, reward, nxt, float(term)))   # store transition
        state, ep_return, steps = nxt, ep_return + reward, steps + 1
        learn()
        if steps % target_sync == 0:                     # periodically sync target net
            q_targ.load_state_dict(q.state_dict())
    if ep % 20 == 0:
        print(f"episode {ep:3d}  return {ep_return:6.1f}  eps {eps:.2f}")
# Return climbs from ~20 toward the 500 cap as the pole is balanced longer and longer.

## Visualize the data & results

_Does the agent actually learn? Plot episode return over training — it should rise and then solve the task._

In [ ]:
import numpy as np
# Tiny tabular Q-learning on a CartPole-style balance proxy: a faithful stand-in for
# DQN on CartPole — the LEARNING CURVE has the same shape (rise, then solve/plateau).
rng = np.random.default_rng(0)
N_STATES, CENTER, MAX_STEPS = 11, 5, 200      # tilt -5..+5 mapped to 0..10; center balanced
Q = np.zeros((N_STATES, 2))                    # 2 actions: 0 = push left, 1 = push right
gamma, alpha = 0.99, 0.1
returns = []
for ep in range(600):
    tilt, total = CENTER, 0.0
    eps = max(0.02, 1.0 - ep / 250.0)          # epsilon decay: explore -> exploit
    for t in range(MAX_STEPS):
        a = int(np.argmax(Q[tilt])) if rng.random() >= eps else rng.integers(2)
        push = -1 if a == 0 else 1             # the chosen push
        drift = rng.integers(-1, 2)            # random environment drift in {-1,0,1}
        tilt2 = tilt + push + drift
        fell = tilt2 < 0 or tilt2 >= N_STATES  # fell off -> episode ends
        tilt2 = min(N_STATES - 1, max(0, tilt2))
        r = 1.0 if not fell else 0.0           # +1 per surviving step
        target = r + (0.0 if fell else gamma * Q[tilt2].max())   # Bellman TD target
        Q[tilt, a] += alpha * (target - Q[tilt, a])              # TD update
        tilt, total = tilt2, total + r
        if fell:
            break
    returns.append(total)

sm = np.convolve(returns, np.ones(30) / 30, mode='valid')        # smoothed learning curve
idx = np.linspace(0, len(sm) - 1, 16).astype(int)               # subsample to 16 points
print([[int(i), round(float(sm[i]), 2)] for i in idx])

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The online net guesses $\hat Q(s,a;\theta) = 2$. After acting, $r = 0$ and the target network's best next value is $\max_{a'}\hat Q(s',a';\theta^-) = 5$, with $\gamma = 0.8$. What is the target $y$, and which way does training move $\hat Q(s,a;\theta)$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Plug into $y = r + \gamma\max_{a'}\hat Q(s',a';\theta^-)$. — _The target is reward-now plus the discounted best next value, taken from the FROZEN target net._
- Compute $y = 0 + 0.8\times 5 = 4$. — _The discount $\gamma=0.8$ shrinks the next value from $5$ to $4$._
- Compare $y$ to the current guess: $\delta = y - \hat Q = 4 - 2 = 2 \gt 0$. — _A positive TD error means the guess is too low._

**Answer:** $y = 4$. Since $\delta = 2 \gt 0$, training nudges $\hat Q(s,a;\theta)$ UP from $2$ toward $4$. The target $4$ stays fixed during the update because it was built from $\theta^-$.

</details>

**Problem 2.** Why does plain online Q-learning with a neural network (no replay buffer, no target network) tend to diverge?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Name the three ingredients present. — _Function approximation (the net), bootstrapping (target uses $\hat Q$ of the next state), and off-policy learning (the $\max$) &mdash; the deadly triad._
- Note that consecutive samples are highly correlated. — _Without a replay buffer the net trains on a stream of near-identical frames, biasing the gradient._
- Note the target uses the live weights. — _Without a target net, updating $\theta$ also moves the target $y$, creating a feedback loop._

**Answer:** The deadly triad (function approximation + bootstrapping + off-policy) plus correlated samples and a moving target form a feedback loop that lets value estimates explode. Experience replay de-correlates the samples and the target network freezes the bootstrap, breaking the loop.

</details>

**Problem 3.** In Double DQN, the target is $y = r + \gamma\,\hat Q\big(s', \arg\max_{a'}\hat Q(s',a';\theta);\ \theta^-\big)$. Which net SELECTS the next action and which net EVALUATES it, and what bias does this remove?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read the $\arg\max$ term. — _It uses $\theta$ (the online net) to pick which next action looks best._
- Read the outer $\hat Q(\dots;\theta^-)$. — _It uses $\theta^-$ (the target net) to score that chosen action._
- Recall standard DQN uses $\max_{a'}\hat Q(s',a';\theta^-)$ for both at once. — _Selecting and evaluating with the same noisy max over-estimates value._

**Answer:** The ONLINE net ($\theta$) selects the action; the TARGET net ($\theta^-$) evaluates it. Decoupling them removes the maximization / OVER-ESTIMATION bias that standard DQN's single $\max$ introduces.

</details>